In [ ]:
import marimo as mo
import mne
import numpy as np
from glob import glob
import matplotlib.pyplot as plt
# plt.rcParams.update({"figure.dpi": 300})
import os
import librosa
from cleese_stim.engines import PhaseVocoder
from cleese_stim.engines.phase_vocoder import utils as pvutils
from sklearn.preprocessing import StandardScaler
from mtrf import TRF
from mtrf.stats import neg_mse, pearsonr, nested_crossval
import pandas as pd
import seaborn as sns
import eelbrain as eb
eb.configure(frame=False)
from scipy import stats
from scipy.signal import resample
from tqdm import tqdm
import warnings

# ERP analysis

## Pipeline

### Functions to clean up event markers

In [ ]:
def events_without_python(raw):
    events_from_annot, event_dict = mne.events_from_annotations(raw)

    # Get indices of R11 events with event code 1011 or New Segment indices with event code 99999 and delete them
    useless_events = list(filter(lambda i:
                                 events_from_annot[:, 2][i] == 99999 or 
                                 events_from_annot[:, 2][i] == 1011 or 
                                 events_from_annot[:, 2][i] == 2 or 
                                 events_from_annot[:, 2][i] == 6, 
                                 range(len(events_from_annot[:, 2]))
                                ))
    events = np.delete(events_from_annot, useless_events, 0)

    for i in range(len(events[:, 2])):
        if events[:, 2][i] == 1004 or events[:, 2][i] == 1008:
            events[:, 2][i] = 1001
        elif  events[:, 2][i] == 1012:
            events[:, 2][i] = 1002
        elif events[:, 2][i] == 1024:
            events[:, 2][i] = 1003
        elif events[:, 2][i] == 1028:
            events[:, 2][i] = 1004

    return events

In [ ]:
def events_with_python(raw):
    events_from_annot, event_dict = mne.events_from_annotations(raw)

    # Get indices of R11 events with event code 1011 or New Segment indices with event code 99999 and delete them
    useless_events = list(filter(lambda i: 
                                 events_from_annot[:, 2][i] == 99999 or 
                                 events_from_annot[:, 2][i] == 1011, 
                                 range(len(events_from_annot[:, 2]))
                                ))
    events = np.delete(events_from_annot, useless_events, 0)

    # Get indices of consective equal events and keep only the 1st one and delete the others
    consecutive_equal_events = list(filter(lambda i: events[:, 2][i] == events[:, 2][i+1], range(len(events[:, 2])-1)))
    consecutive_equal_events = [index+1 for index in consecutive_equal_events]
    events = np.delete(events, consecutive_equal_events, 0)

    # If experiment starts with an S event with event code 2, delete it
    if events[:, 2][0] == 2:
        events = np.delete(events, 0, 0)

    # Change S event so it takes event code from R events (for e.g.- if R3 is followed by S2, change it to S3)
    for i in range(0, len(events), 2):
        events[i+1][2] = events[i][2]

    # Extract S events into different array
    s_events = events[1::2]

    return s_events

### Functions to preprocess data

In [ ]:
def create_stim_channel(raw, events):
    raw.load_data()
    stim_data = np.zeros((1, len(raw.times)))

    # Add stimulus channel in 'raw' object's info class
    info = mne.create_info(["STI"], raw.info["sfreq"], ["stim"])
    stim_raw = mne.io.RawArray(stim_data, info)
    raw.add_channels([stim_raw], force_update_info=True)

    # Add events extracted from annotations to the stimulus channel
    raw.add_events(events, stim_channel="STI")

In [ ]:
def crop_data(raw, t_from, t_to):
    first_event_time = mne.find_events(raw)[0][0]
    last_event_time = mne.find_events(raw)[-1][0]

    part_to_remove_from_beginning = (first_event_time - abs(t_from*500))/1000
    part_to_remove_from_end = (last_event_time + abs(t_to*5000))/1000
    raw.crop(part_to_remove_from_beginning, part_to_remove_from_end)

In [ ]:
def apply_filter(raw):
    # Soft bandpass Butterworth filter 
    iir_params = mne.filter.construct_iir_filter(
        dict(order=2, ftype="butter", output="sos"), 
        f_pass=[0.1, 30], 
        f_stop=None, 
        sfreq=1000, 
        btype="bandpass", 
        return_copy=False,
        verbose=False
    )
    raw.filter(0.1, 30, method="iir", iir_params=iir_params)

    # Notch filter
    raw.notch_filter(reqs=np.arange(50, 251, 50), method="fir", fir_design="firwin2")

In [ ]:
def set_reference(raw, ref):
    mne.add_reference_channels(raw, ref, copy=False)
    mne.set_eeg_reference(raw, ref_channels="average", projection=True)
    raw.apply_proj()

In [ ]:
def run_ica(epochs, eog_proxy):
    ica = mne.preprocessing.ICA(random_state=99)
    ica.fit(epochs)

    # Find which ICs match the EOG pattern
    eog_indices, eog_scores = ica.find_bads_eog(epochs, ch_name=eog_proxy)
    print(f"**************** Automatically found EOG artifact ICA components: {eog_indices} ****************")

    # # Find which ICs match the EMG pattern
    # muscle_idx_auto, scores = ica.find_bads_muscle(epochs[~reject_log.bad_epochs])
    # print(f'**************** Automatically found muscle artifact ICA components: {muscle_idx_auto} ****************')

    ica.exclude = eog_indices

    ica.plot_overlay(epochs.average(), exclude=ica.exclude)
    ica.apply(epochs)

In [ ]:
def compute_grand_average(dir, conditions, channels_for_viz):
    evokeds_per_condition = {}
    for cond in conditions:
        test = []
        for file in glob(f"{dir}/*/*_ave.fif"):
            evk = mne.read_evokeds(file, condition=cond)
            test.append(evk)
        evokeds_per_condition[condition_dict[cond]["name"]] = mne.combine_evoked(test, weights='nave')

    # save combined evoked objects of all conditions into file
    mne.write_evokeds(f"{dir}/grand_ave.fif", list(evokeds_per_condition.values()), overwrite=True)

    for channel in channels_for_viz:
        fig = mne.viz.plot_compare_evokeds(
            evokeds_per_condition,
            picks=channel, 
            combine=None, 
            time_unit="ms", 
            ylim=dict(eeg=[-5, 5]), 
            invert_y=True,
            colors=dict(standard="blue", neutral="green", deviant1="black", deviant2="red"), 
            styles={
                "standard": {"linewidth": 1}, "neutral": {"linewidth": 1}, 
                "deviant1": {"linewidth": 1}, "deviant2": {"linewidth": 1}
            }
        )
        fig[0].savefig(f'{dir}/grand_ave_{channel}.png')

### Main function

In [ ]:
def run_pipeline(
    vhdr_file, channels, montage, epoch_limits, baseline, 
    subjs_with_presentation_markers, channels_for_viz,
    output_dir="./analysis"
):

    subject = os.path.basename(vhdr_file).split(".")[0]
    print(subject)

    """
    I/0
    """
    raw = mne.io.read_raw_brainvision(vhdr_file, verbose=False)
    mne.rename_channels(raw.info, mapping=dict(zip(raw.ch_names, channels)))
    raw.set_montage(montage)


    """
    ADD MARKERS
    """
    if subject in subjs_with_presentation_markers:
        events = events_without_python(raw)
    else:
        events = events_with_python(raw)
    # create a 'STIM' channel for the event markers and add the fixed markers to it
    create_stim_channel(raw, events)


    """
    CROP & FILTER RAW DATA
    """
    crop_data(raw, epoch_limits[0], epoch_limits[1])
    # apply bandpass and notch filters
    apply_filter(raw)


    """
    SET REFERENCE
    """
    set_reference(raw, ref="Cz")
    # set montage again after applying reference projection to the data
    raw.set_montage(montage)


    """
    CREATE EPOCHS
    """
    epochs = mne.Epochs(raw, events, tmin=epoch_limits[0], tmax=epoch_limits[1], preload=True, baseline=None)


    """
    ICA
    """
    # global rejection threshold to remove before ICA (improves ICA performance)
    # reject = get_rejection_threshold(epochs, decim=2)
    # fit and apply ICA
    run_ica(epochs, "Fp1")
    epochs.apply_baseline(baseline=baseline)


    """
    AUTOREJECT
    """
    # interpolate and repair epochs after ICA
    # epochs_ar = AutoReject(random_state=11, n_jobs=1, verbose=True).fit_transform(epochs)


    """
    WRITE EPOCHS TO FILE
    """
    epochs.save(f"{output_dir}/{subject}/{subject}_epo.fif", overwrite=True)


    """
    COMPUTE ERPs
    """
    # create Evoked object from epochs (an Evoked object contains the average data over all epochs)
    evoked_standard = epochs["1001"].average()
    evoked_neutral = epochs["1002"].average()
    evoked_deviant1 = epochs["1003"].average()
    evoked_deviant2 = epochs["1004"].average()

    mne.write_evokeds(
        f"{output_dir}/{subject}/{subject}_ave.fif", 
        [evoked_standard, evoked_neutral, evoked_deviant1, evoked_deviant2], 
        overwrite=True
    )

    evokeds = dict(standard=evoked_standard, neutral=evoked_neutral, deviant1=evoked_deviant1, deviant2=evoked_deviant2)


    """
    VISUALIZE ERPs
    """
    for channel in channels_for_viz:
        fig = mne.viz.plot_compare_evokeds(
            evokeds, 
            picks=channel, 
            combine=None, 
            time_unit="ms", 
            ylim=dict(eeg=[-10, 10]), 
            invert_y=True,
            colors=dict(standard="blue", neutral="green", deviant1="black", deviant2="red"), 
            styles={
                "standard": {"linewidth": 1}, "neutral": {"linewidth": 1}, 
                "deviant1": {"linewidth": 1}, "deviant2": {"linewidth": 1}
            }
        )
        fig[0].savefig(f"{output_dir}/{subject}/{subject}_{channel}.png")

## Rise/fall

In [ ]:
channel_names = [
    "Fp1","Fz","F3","F7","FT9","FC5","FC1","C3",
    "T7","TP9","CP5","CP1","Pz","P3","P7","O1",
    "Oz","O2","P4","P8","TP10","CP6","CP2","C4",
    "T8","FT10","FC6","FC2","F4","F8","Fp2", "AF7",
    "AF3","AFz","F1","F5","FT7","FC3","C1","C5",
    "TP7","CP3","P1","P5","PO7","PO3","POz","PO4",
    "PO8","P6","P2","CPz","CP4","TP8","C6","C2",
    "FC4","FT8","F6","AF8","AF4","F2","FCz", "Cz"
]
# channels_to_visualize = ['Fz', 'Pz', 'Oz', 'AFz', 'POz', 'CPz', 'FCz', 'Cz']

In [ ]:
# for file in sorted(glob("./eeg_data/rise/*.vhdr")):
#     run_pipeline(
#         vhdr_file=file,
#         channels=channel_names,
#         montage=mne.channels.make_standard_montage("easycap-M1"),
#         epoch_limits=[-0.1, 0.6],
#         baseline=(-0.1, 0),
#         subjs_with_presentation_markers=["0001", "0002", "0003"],
#         channels_for_viz=["Pz"],
#         output_dir="./analysis_new/"
#     )

In [ ]:
# compute_grand_average(
#     dir="./analysis_new", 
#     conditions=["standard", "neutral", "rise", "fall"], 
#     channels_for_viz=["Pz"]
# )

## Smile/rough

In [ ]:
# for file in sorted(glob("./eeg_data/smile/*.vhdr")):
#     run_pipeline(
#         vhdr_file=file,
#         channels=channel_names,
#         montage=mne.channels.make_standard_montage("easycap-M1"),
#         epoch_limits=[-0.1, 0.6],
#         baseline=(-0.1, 0),
#         subjs_with_presentation_markers=["0003"],
#         channels_for_viz=["Pz"],
#         output_dir="./analysis_smile/"
#     )

In [ ]:
# compute_grand_average(
#     dir="./analysis_smile", 
#     conditions=["standard", "neutral", "smile", "rough"], 
#     channels_for_viz=["Pz"]
# )

# TRF analysis

In [ ]:
def subj_from_file(file):
    start_idx = file.find("00")
    return file[start_idx:start_idx+4]

def deviant_from_file(file):
    return file.split("/")[-1].split("_")[1][:-4]

def info_from_file(file):
    subj = subj_from_file(file)

    if "rise" in file.lower():
        paradigm = "rise"
    elif "smile" in file.lower():
        paradigm = "smile"

    name = os.path.basename(glob(f"./sounds/{paradigm}/{subj}/*.wav")[0]).split("_")[0]

    return subj, name, paradigm

def get_deviant_name(id, paradigm):
    if id == "1001":
        condition = "standard"
    elif id == "1002":
        condition = "neutral"
    elif id == "1003":
        condition = "rise" if paradigm == "rise" else "smile"
    elif id == "1004":
        condition = "fall" if paradigm == "rise" else "rough"

    return condition

## Extract input feature from sound

In [ ]:
def extract_feature(sound_file, feature, n_fft=2048, frame_length=2048, hop_length=128):
    # some sound files have sampling rate != 22050
    y, original_sr = librosa.load(sound_file, sr=22050)

    match feature:
        case "rms":
            data = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0]
        case "pitch":
            data, voiced_flag, voiced_probs = librosa.pyin(
                y=y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'), 
                sr=original_sr, frame_length=frame_length, hop_length=hop_length
            )
            # interpolate (copied from Cleese)
            start_value = end_value = np.mean(data[np.nonzero(~np.isnan(data))])
            data = pvutils.interpolate_series(data, start_value = start_value, end_value = end_value)
        case "spectral_centroid":
            data = librosa.feature.spectral_centroid(y=y, sr=original_sr, n_fft=n_fft, hop_length=hop_length)[0]
        case "spectral_flatness":
            data = librosa.feature.spectral_flatness(y=y, n_fft=n_fft, hop_length=hop_length)[0]

    return data.flatten()

## Get outputs (epoch data)

In [ ]:
def get_epochs(epoch_file, tmin, tmax, channels=[]):
    epochs = mne.read_epochs(epoch_file, preload=True, verbose=False)
    epochs = epochs.drop_channels('STI')
    epochs = epochs.crop(tmin, tmax)

    if channels != []:
        epochs = epochs.pick(channels)

    return epochs

## Input checks

In [ ]:
def plot_sounds(file, axes=None, **kwargs):
    subj, name, paradigm = info_from_file(file)

    if axes is None:
        fig, ax = plt.subplots(figsize=(6 ,6))
    else:
        fig, ax = None, axes

    if paradigm == "rise":
        p1 = extract_feature(glob(f"./sounds/{paradigm}/{subj}/*_rise.wav")[0], **kwargs)
        p2 = extract_feature(glob(f"./sounds/{paradigm}/{subj}/*_fall.wav")[0], **kwargs)
        ax.plot(p1, label="rise")
        ax.plot(p2, label="fall")
    elif paradigm == "smile":
        p1 = extract_feature(glob(f"./sounds/{paradigm}/{subj}/*_smile.wav")[0], **kwargs)
        p2 = extract_feature(glob(f"./sounds/{paradigm}/{subj}/*_rough.wav")[0], **kwargs)
        ax.plot(p1, label="smile")
        ax.plot(p2, label="rough")

    neutral = extract_feature(glob(f"./sounds/{paradigm}/{subj}/*_neutral.wav")[0], **kwargs)
    ax.plot(neutral, label="neutral")
    ax.set_title(f"{subj} - {name}")
    ax.legend()

    return plt.gcf()

In [ ]:
# for subject_file in sorted(glob("./analysis/rise/*/")):
#     fig, axs = plt.subplots(ncols=2, figsize=(10, 4))
#     plt.tight_layout()
#     plot_sounds(subject_file, feature="rms", axes=axs[0])
#     plot_sounds(subject_file, feature="pitch", frame_length=2048, hop_length=int(22050/128), axes=axs[1])
#     plt.show()

# for subject_file in sorted(glob("./analysis/smile/*/")):
#     fig, axs = plt.subplots(ncols=2, figsize=(10, 4))
#     plt.tight_layout()
#     plot_sounds(subject_file, feature="spectral_centroid", n_fft=2048, hop_length=int(22050/128), axes=axs[0])
#     plot_sounds(subject_file, feature="spectral_flatness", n_fft=2048, hop_length=int(22050/128), axes=axs[1])
#     plt.show()

## Prepare and format data for TRF

In [ ]:
def prepare_data(epoch_file, sr, features=[]):
    id, name, paradigm = info_from_file(epoch_file)
    # load epochs of deviant stimuli
    epochs = get_epochs(epoch_file, tmin=0, tmax=0.6, channels=["Pz"])[["1002", "1003", "1004"]]

    conditions, stims, resps = [], [], []
    for i in tqdm(range(len(epochs))):
        event_id = list(epochs[i].event_id.keys())[0]
        condition = get_deviant_name(event_id, paradigm)
        sound_file = f"./sounds/{paradigm}/{id}/{name}_{condition}.wav"

        if len(features) == 1:
            stim = extract_feature(sound_file, features[0], frame_length=2048, n_fft=2048, hop_length=int(22050/sr))
        elif len(features) > 1:
            s = []
            for feature in features:
                s.append(extract_feature(sound_file, feature, frame_length=2048, n_fft=2048, hop_length=int(22050/sr)))
            stim = np.vstack(s).T

        stim_sr = len(stim)/librosa.get_duration(path=sound_file)

        resp = epochs[i].resample(sfreq=stim_sr)
        resp = resp.get_data().flatten()
        # sometimes len(resp) is 77 instead of 78 so pad
        resp = np.pad(resp, (0, 78-len(resp)), mode="edge")

        # o3
        if len(stim) < len(resp):
            warnings.warn(f"{id}: Stimulus ({len(stim)}) shorter than response ({len(resp)}). Zero-padding stimulus...")
            if len(stim.shape) == 1:
                stim = np.pad(stim, (0, len(resp)-len(stim)), mode="constant")
            elif len(stim.shape) == 2:
                stim = np.pad(stim, ((0, len(resp)-len(stim)), (0, 0)), mode="constant")
        elif len(stim) > len(resp):
            warnings.warn(f"{id}: Stimulus ({len(stim)}) longer than response ({len(resp)}). Cropping stimulus...")
            stim = stim[:len(resp)]

        conditions.append(condition)
        # features.append("-".join(x for x in features))
        stims.append(stim)
        resps.append(resp)

    df = pd.DataFrame({"condition": conditions, "stimulus": stims, "response": resps})
    df["id"] = id
    df["name"] = name

    return df

In [ ]:
# rise_dfs = []
# for epoch_file in sorted(glob("./analysis/rise/*/*_epo.fif")):
#     rise_subjData = prepare_data(epoch_file, sr=128, features=["rms", "pitch"])
#     rise_dfs.append(rise_subjData)

# rise_df = pd.concat(rise_dfs)
# rise_df["stimulus"] = rise_df.stimulus.transform(lambda x: x.astype(dtype="float64"))
# # rise_df.to_parquet("./analysis/rise/rise_rms_pitch_test.gzip", index=False)
# rise_df.to_pickle("./analysis/rise/rise_rms_pitch.pkl")
# rise_df

# smile_dfs = []
# for epoch_file in sorted(glob("./analysis/smile/*/*_epo.fif")):
#     smile_subjData = prepare_data(epoch_file, sr=128, features=["spectral_centroid", "spectral_flatness"])
#     smile_dfs.append(smile_subjData)

# smile_df = pd.concat(smile_dfs)
# smile_df["stimulus"] = smile_df.stimulus.transform(lambda x: x.astype(dtype="float64"))
# # smile_df.to_parquet("./analysis/smile/smile_spectralFlatness.gzip", index=False)
# smile_df.to_pickle("./analysis/smile/smile_sc_sf.pkl")
# smile_df

## Train TRF

In [ ]:
def train_trf(data, training_condition, trf_maxLag, sr):
    ids, trfs = [], []

    X = data.loc[data.condition==training_condition].reset_index(drop=True)
    stimulus = X.stimulus.to_list()
    response = X.response.to_list()

    trf = TRF(direction=1, metric=pearsonr)
    r_unbiased, best_regularization = nested_crossval(
        trf, stimulus, response,
        fs=sr, tmin=0, tmax=trf_maxLag, regularization=np.logspace(-1, 6, 20), k=5
    )
    # best_regularization = 1
    trf.train(
        stimulus=stimulus, response=response,
        fs=sr, tmin=0, tmax=trf_maxLag, regularization=best_regularization
    )

    ids.append(data.id.iloc[0])
    trfs.append(trf)

    return pd.DataFrame({"id": ids, "trf": trfs})

In [ ]:
# Smile/rough
# Spectral flatness of smile is lower than other deviants for: 
# 0008 (Mathilde), 0030 (Lucas), 0032 (Nassim), 0033 (Lucille), 0034 (Fabien)

In [ ]:
rise_data = pd.read_pickle("./analysis/smile/smile_sc_sf.pkl").reset_index(drop=True)
# for Rise/fall
# exclude = [
#     "0008",
#     "0038"
# ]
# for Smile/rough
exclude = [
    "0008",
    "0023",
    "0030",
    "0032", 
    "0033", 
    "0034", 
]

trf_dfs = []
for id in rise_data.id.unique():
    if id in exclude:
        pass
    else:
        trf_df = train_trf(
            data=rise_data.loc[rise_data.id==id], training_condition="neutral", 
            trf_maxLag=0.6, sr=128
        )
        trf_dfs.append(trf_df)

rise_trfs = pd.concat(trf_dfs).reset_index(drop=True)
rise_trfs

In [ ]:
def format_for_plot(data, normalize_weights=False):
    data["times"], data["weights"] = zip(*data.trf.apply(lambda x: (x.times, x.weights)))

    n_cols = len(data["weights"].iloc[0])
    new_cols = [f"weights{i+1}" for i in range(n_cols)]
    data[new_cols] = pd.DataFrame(list(zip(*data["weights"].values))).T
    data = data.drop(["trf", "weights"], axis=1)

    # if normalize_weights:
    #     data["weights"] = data.weights.apply(
    #         lambda x: x/max(abs(x))
    #     )

    data = data.explode(column=list(data.columns)[1:]).explode(column=new_cols).reset_index(drop=True)

    return data

In [ ]:
# for subject_id in rise_trfs.id.unique():
#     subj_trf = format_for_plot(rise_trfs.loc[rise_trfs.id==subject_id], normalize_weights=False)
#     sns.lineplot(
#         data=subj_trf, 
#         y="weights", x="times", c="forestgreen", lw=5
#     )
#     plt.axhline(y=0, ls="--", c="k", alpha=0.5)
#     plt.xlabel("Time [s]")
#     plt.ylabel("TRF weights")
#     plt.title(subject_id)
#     plt.show()

In [ ]:
trf = format_for_plot(rise_trfs, normalize_weights=False)

sns.lineplot(data=trf, y="weights1", x="times", c="forestgreen", lw=5)
sns.lineplot(data=trf, y="weights2", x="times", c="forestgreen", lw=5)
plt.axhline(y=0, ls="--", c="k", alpha=0.5)
plt.xlabel("Time [s]")
plt.ylabel("TRF weights")

## Predict responses using TRFs

In [ ]:
def trf_predict(d1, d2):
    data = pd.merge(d1, d2, on=["id"])
    # use TRFs to predict responses and compute correlation between actual and predicted response
    data[["prediction", "correlation"]] = data.apply(
        lambda x: x.trf.predict(x.stimulus, x.response.flatten()), 
        axis=1, result_type="expand"
    )
    data["prediction"] = data.prediction.transform(lambda x: x[0].flatten())

    return data

In [ ]:
rise_df = trf_predict(rise_data, rise_trfs)
rise_df

In [ ]:
# test = rise_df.loc[(rise_df.condition=="fall") & (rise_df.feature=="pitch")]
# test = test.loc[:, ["id", "times", "stimulus", "response", "prediction"]]
# test = test.explode(column=["times", "stimulus", "response", "prediction"]).reset_index(drop=True)

# for subj_id in test.id.unique():
#     sub_data = test.loc[test.id==subj_id]

#     fig, axs = plt.subplots(ncols=3, figsize=(12, 4))
#     plt.tight_layout()

#     sns.lineplot(data=sub_data, x="times", y="stimulus", ax=axs[0])
#     axs[0].set_title(f"{subj_id}: [{min(sub_data.stimulus):.2f}, {max(sub_data.stimulus):.2f}]")

#     sns.lineplot(data=sub_data, x="times", y="response", ax=axs[1])
#     axs[1].set_title(f"[{min(sub_data.response):.1e}, {max(sub_data.response):.1e}]")

#     sns.lineplot(data=sub_data, x="times", y="prediction", ax=axs[2])    
#     axs[2].set_title(f"[{min(sub_data.prediction):.1e}, {max(sub_data.prediction):.1e}]")

#     plt.show()

In [ ]:
sns.barplot(
    data=rise_df, x="condition", y="correlation", order=["neutral", "smile", "rough"],
    hue="condition", hue_order=["neutral", "smile", "rough"], palette=["r", "royalblue", "forestgreen"]
)
plt.xlabel("Feature")
plt.ylabel("Pearson r")

## Temporal cluster permutation test

In [ ]:
def cluster_permutation(data, prediction_condition, color):
    data = data.loc[
        data.condition==prediction_condition, 
        ["id", "times", "response", "prediction"]
    ]
    data = data.melt(id_vars=["id", "times"], value_vars=["response", "prediction"])
    ds = make_eelbrain_dataset(data)
    print(ds.summary())

    res_timecoure = eb.testnd.TTestRelated(
        'pz', 'condition', 'response', 'prediction', match='id', data=ds,
        pmin=0.05,
        tstart=0,
        tstop=0.600,
        mintime=0.1
    )
    clusters = res_timecoure.find_clusters(0.05)
    print(clusters)

    p = eb.plot.UTSStat(
        'pz', 'condition', match='id', error="ci", within_subject_error=False, data=ds, run=False,
        ylabel="Pz Amplitude [μV]", colors= {
            ("response"): eb.plot.Style(color="k", linestyle='-', linewidth=4),
            ("prediction"): eb.plot.Style(color=color, linestyle='-', linewidth=4),
        }, legend=False, h=4, w=6,
    )
    # p.set_ylim(-2e-6, 6e-6)
    p.set_clusters(clusters)
    plt.axhline(y=0, ls="--", c="k", alpha=0.5)
    plt.legend(frameon=False)

    return plt.gcf()

In [ ]:
def make_eelbrain_dataset(data):
    rows = []
    for idx, row in data.iterrows():
        subject = eb.Var(int(row.id))
        # Factor needs to be array-like of values
        condition = eb.Factor([row.variable])
        time = eb.UTS(tmin=0, tstep=row.times[1]-row.times[0], nsamples=len(row.times))
        pz_data = eb.NDVar(row.value, dims=(time,))

        case = [subject, condition, pz_data]
        rows.append(case)

    ds = eb.Dataset().from_caselist(names=["id", "condition", "pz"], cases=rows)

    return ds

In [ ]:
tcp_df = rise_df.loc[:, ["id", "condition", "times", "stimulus", "response", "prediction"]]
tcp_df = tcp_df.explode(column=["times", "stimulus", "response", "prediction"]).reset_index(drop=True)
tcp_df = tcp_df.groupby(["id", "condition", "times"]).mean().reset_index()
tcp_df = tcp_df.groupby(["id", "condition"])[["times", "stimulus", "response", "prediction"]].agg(list).reset_index()
# tcp_df["response"] = tcp_df.response.apply(zscore)
# tcp_df["prediction"] = tcp_df.prediction.apply(zscore)
tcp_df

In [ ]:
cluster_permutation(data=tcp_df, prediction_condition="rough", color="forestgreen")

## Statistical tests between TRF fits

In [ ]:
import pingouin as pg

In [ ]:
pd.concat([
    pg.ttest(
        x=rise_df.loc[rise_df.condition=="neutral"].correlation,
        y=rise_df.loc[rise_df.condition=="rise"].correlation,
        paired=True
    ),
    pg.ttest(
        x=rise_df.loc[rise_df.condition=="neutral"].correlation,
        y=rise_df.loc[rise_df.condition=="fall"].correlation,
        paired=True
    ),
    pg.ttest(
        x=rise_df.loc[rise_df.condition=="rise"].correlation,
        y=rise_df.loc[rise_df.condition=="fall"].correlation,
        paired=True
    )
])

In [ ]:
pd.concat([
    pg.ttest(
        x=rise_df.loc[rise_df.condition=="neutral"].correlation,
        y=rise_df.loc[rise_df.condition=="smile"].correlation,
        paired=True
    ),
    pg.ttest(
        x=rise_df.loc[rise_df.condition=="neutral"].correlation,
        y=rise_df.loc[rise_df.condition=="rough"].correlation,
        paired=True
    ),
    pg.ttest(
        x=rise_df.loc[rise_df.condition=="smile"].correlation,
        y=rise_df.loc[rise_df.condition=="rough"].correlation,
        paired=True
    )
])